<div align="center">

<a href="https://colab.research.google.com/github/utkukose/human-computer-interaction-NB-lecture/blob/main/exams/final/FIN3_mobile_voice_and_suggestions.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"></a>

# Final option 3: A mobile assistant with voice and suggestions

**Human-Computer Interaction (BIL-429 İnsan-Bilgisayar Etkileşimi)**<br>Final project: An interface with an artificial intelligence (AI) feature, designed and evaluated. Starter Colab notebook

[Project page](https://github.com/utkukose/human-computer-interaction-NB-lecture/tree/main/exams/final) | [Course repository](https://github.com/utkukose/human-computer-interaction-NB-lecture)

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

<sub>Content licensed under CC BY 4.0, code under MIT. This course is updated in line with current developments in the field. First version: November 2026.</sub>

</div>

## How to use this starter Colab notebook

Save a copy in Drive first, with File, Save a copy in Drive. Every code cell runs as it is with an example case: A to-do app with voice commands and an assistant that suggests when to remind its user. Replace the values marked `# TODO` with the measurements of your own interface. Then run all cells again from the top, with Runtime, Run all, and use the printed values and figures in the report. Each section names the week and the lecture section that explain its method. The last section collects the numbers for the results table of the report. Keep your own data in the cells, so that the Colab notebook documents how every number of the report was obtained.

## 0. Setup

The cell loads the libraries and creates the dictionary `RESULTS`, which collects the numbers of the report.

In [ ]:
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

RANDOM_STATE = 42
RESULTS = {}            # the numbers for the results table of the report
print("Setup complete.")

## 1. Voice commands and a read-back

*Lecture: [Week 11, Speech and multimodal interaction](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#speech-and-multimodal)*

The word error rate (WER) is $WER = (S + D + I)/N$: the substitutions, deletions and insertions that turn the spoken command into the recognised one, divided by its $N$ words. With independent errors, a command of $n$ words is recognised completely with $(1 - w)^n$. A read-back that the user hears before the app acts catches a share $c$ of the wrong commands. In the example, a to-do app takes commands by voice. Speak at least 20 commands to your app, and write what it recognised.

In [ ]:
def msd(a, b):
    """Minimum string distance: the fewest edits that turn a into b."""
    D = np.zeros((len(a) + 1, len(b) + 1), dtype=int)
    D[:, 0] = np.arange(len(a) + 1)
    D[0, :] = np.arange(len(b) + 1)
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            D[i, j] = min(D[i - 1, j] + 1, D[i, j - 1] + 1, D[i - 1, j - 1] + cost)
    return int(D[-1, -1])

def wer(ref, hyp):
    """Word error rate: word edits divided by the words of the reference."""
    r, h = ref.split(), hyp.split()
    return msd(r, h) / len(r)

TESTS = [   # TODO: what you said, and what the app recognised
    ("remind me to call the dentist at five",
     "remind me to call the dentist at five"),
    ("add milk and eggs to the shopping list",
     "add milk and eggs to the shopping list"),
    ("move the meeting to thursday morning",
     "move the meeting to tuesday morning"),
    ("remind me to water the plants tonight",
     "remind me to water the plans tonight"),
    ("mark the report as done", "mark the report as done"),
    ("delete the reminder about the gym", "delete the reminder about the gym"),
]
errors = sum(msd(r.split(), h.split()) for r, h in TESTS)
words = sum(len(r.split()) for r, _ in TESTS)
w = errors / words
wrong = np.mean([r != h for r, h in TESTS])
C_READBACK = 0.9                 # TODO: share of wrong commands that users catch
print(f"WER {w:.3f} over {words} words; commands wrong {wrong:.3f}")
print(f"a command of 7 words is right with {(1 - w) ** 7:.3f}")
print(f"wrong commands acted on: {wrong:.3f} without, "
      f"{wrong * (1 - C_READBACK):.3f} with a read-back")
RESULTS["word error rate"] = round(w, 3)
RESULTS["wrong commands acted on, with a read-back"] = round(
    wrong * (1 - C_READBACK), 3)

## 2. Notifications and focus

*Lecture: [Week 11, Notifications](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-11/lecture.html#notifications)*

Notifications that arrive at random moments with the rate $\lambda$ per hour leave a task of $T$ hours free of interruptions with the probability $P_0 = e^{-\lambda T}$. If the app collects them and delivers them every $B$ hours, a task is interrupted with $P_B = \frac{T}{B}(1 - e^{-\lambda B})$ when $T \le B$. Count the notifications of your app over a few days.

In [ ]:
LAMBDA = 2.4      # TODO: notifications of the app per hour while in use
T_TASK = 25 / 60  # TODO: a task of focused work (hours)
B = 1.0           # TODO: the interval of a batched delivery (hours)
p0 = math.exp(-LAMBDA * T_TASK)
p_b = T_TASK / B * (1 - math.exp(-LAMBDA * B))
print(f"interrupted at random: {1 - p0:.3f}; with a batch every {B} h: {p_b:.3f}")
RESULTS["interrupted tasks, random"] = round(1 - p0, 3)
RESULTS["interrupted tasks, batched"] = round(p_b, 3)

## 3. Suggestions that users learn to trust or not

*Lecture: [Week 13, Trust over time: Aversion and appreciation](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-13/lecture.html#trust-over-time)*

The assistant suggests when to remind the user of a task. A good suggestion saves the time $t_w - t_s$ of choosing a time by hand, and a bad one costs $t_f$ more, so the suggestions pay off above the break-even reliability $p^* = 1 - (t_w - t_s)/t_f$. A simple model of trust rises after each success and falls after each failure; a display of the recent reliability pulls it toward the truth. The example lets the reliability drop after an update in the middle of the period.

In [ ]:
T_S, T_W, T_F = 3, 15, 40     # TODO: s to accept, s to choose by hand, s lost
P_STAR = 1 - (T_W - T_S) / T_F
RELIAB = [0.9] * 10 + [0.6] * 10 + [0.9] * 10   # TODO: reliability per use
rng = np.random.default_rng(RANDOM_STATE)
OUTCOMES = (rng.random(len(RELIAB)) < RELIAB).astype(int)

def trust_run(outcomes, rates=None, t0=0.8, a_s=0.1, a_f=0.4, beta=0.5):
    """Accept when trust > P_STAR; update the trust after each suggestion seen."""
    trust, accepts, trace = t0, [], []
    for t, ok in enumerate(outcomes):
        if rates is not None:
            trust += beta * (rates[t] - trust)    # the display of the reliability
        trace.append(trust)
        accept = trust > P_STAR
        accepts.append(int(accept))
        if accept:                                # only accepted ones are seen
            trust += a_s * (1 - trust) if ok else -a_f * trust
    return np.array(accepts), np.array(trace)

def seconds(accepts):
    """Expected time of all uses: t_s + t_f (1 - p) if accepted, else t_w."""
    return round(sum(T_S + T_F * (1 - p) if a else T_W
                     for a, p in zip(accepts, RELIAB)))

for label, rates in [("no display", None), ("with a display", RELIAB)]:
    accepts, trace = trust_run(OUTCOMES, rates)
    print(f"{label}: accepted {accepts.sum()} of {len(accepts)}, "
          f"expected {seconds(accepts)} s")
    RESULTS[f"expected seconds of {len(RELIAB)} uses, {label}"] = seconds(accepts)
best = [int(p > P_STAR) for p in RELIAB]
print(f"break-even reliability {P_STAR:.2f}; accepting exactly above it: "
      f"{seconds(best)} s")
RESULTS["break-even reliability"] = round(P_STAR, 2)

In [ ]:
# Figure: the trust of the model with and without the display
plt.figure(figsize=(7.5, 3.4))
for label, rates in [("no display", None), ("with a display", RELIAB)]:
    plt.plot(trust_run(OUTCOMES, rates)[1], label=label)
plt.plot(RELIAB, "k:", label="true reliability")
plt.axhline(P_STAR, color="#C98510", ls="--", lw=1, label="break-even")
plt.xlabel("use")
plt.ylabel("trust")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 4. A usability test of voice against touch

*Lecture: [Week 9, Completion rates](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#completion-rates) and [Week 9, Time on task](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#time-on-task) and [Week 9, Comparing designs](https://utkukose.github.io/human-computer-interaction-NB-lecture/weeks/week-09/lecture.html#comparing-designs)*

Each participant adds the same tasks by touch and by voice, in a balanced order, and fills in the System Usability Scale (SUS) for each. The completion rate gets an adjusted Wald interval, the task times a geometric mean, and Welch's test compares the logarithms of the times. The example simulates 10 participants; a test with real participants asks for their consent and keeps the data anonymous.

In [ ]:
def sus_score(answers):
    """SUS score of one questionnaire: ten answers from 1 to 5, statement 1 first."""
    total = 0
    for j, x in enumerate(answers, start=1):
        total += x - 1 if j % 2 == 1 else 5 - x
    return 2.5 * total

def adjusted_wald(x, n, z=1.96):
    """Adjusted Wald interval of a completion rate: (p_adj, low, high)."""
    p_adj = (x + z ** 2 / 2) / (n + z ** 2)
    margin = z * math.sqrt(p_adj * (1 - p_adj) / (n + z ** 2))
    return p_adj, max(0.0, p_adj - margin), min(1.0, p_adj + margin)

def geo_mean_ci(times, conf=0.95):
    """Geometric mean of task times with its t interval from the logarithms."""
    logs = np.log(np.asarray(times, dtype=float))
    n, mean, sd = len(logs), logs.mean(), logs.std(ddof=1)
    t = stats.t.ppf(0.5 + conf / 2, n - 1)
    margin = t * sd / math.sqrt(n)
    return math.exp(mean), math.exp(mean - margin), math.exp(mean + margin)

In [ ]:
# TODO: replace the simulated sessions with yours, one row per person and mode
rng = np.random.default_rng(RANDOM_STATE + 1)
rows = []
MODES = [("touch", 48, 0.95, 3.6), ("voice", 31, 0.85, 3.9)]
for mode, t_med, p_done, sus_mean in MODES:
    for k in range(10):
        sus = np.clip(np.round(rng.normal(sus_mean, 0.6, 10)), 1, 5)
        sus[1::2] = 6 - sus[1::2]                 # even statements are negative
        rows.append(dict(mode=mode, participant=k + 1,
                         completed=int(rng.random() < p_done),
                         time_s=round(t_med * math.exp(rng.normal(0, 0.3)), 1),
                         sus=sus_score(sus)))
SESSIONS = pd.DataFrame(rows)
for mode, part in SESSIONS.groupby("mode"):
    p, lo, hi = adjusted_wald(int(part.completed.sum()), len(part))
    gm, glo, ghi = geo_mean_ci(part.time_s[part.completed == 1])
    print(f"{mode}: completed {part.completed.sum()} of {len(part)} ({lo:.2f} to "
          f"{hi:.2f}), time {gm:.1f} s ({glo:.1f} to {ghi:.1f}), "
          f"SUS {part.sus.mean():.1f}")
    RESULTS[f"{mode}: geometric mean time (s)"] = round(gm, 1)
touch, voice = (SESSIONS[SESSIONS["mode"] == m] for m in ("touch", "voice"))
p_time = stats.ttest_ind(np.log(touch.time_s[touch.completed == 1]),
                         np.log(voice.time_s[voice.completed == 1]),
                         equal_var=False).pvalue
fmt_p = lambda p: f"{p:.3f}" if p >= 0.001 else "< 0.001"
print(f"Welch's test on the logarithms of the times: p = {fmt_p(p_time)}")
RESULTS["p of the times (Welch on logarithms)"] = fmt_p(p_time)

## 5. Results for the report

The cell collects the numbers that the sections above stored in `RESULTS`. Copy them into the results table of the report, and explain each one in the text. A value that changes the design deserves a figure.

In [ ]:
import pandas as pd

print(pd.Series(RESULTS, name="value", dtype=object).to_string())

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>